# Solutions – Day 23 Exercises

In [ ]:
import torch
from diffusers import StableDiffusionImg2ImgPipeline, StableDiffusionInpaintPipeline
from PIL import Image, ImageDraw
import requests
from io import BytesIO
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
img2img = StableDiffusionImg2ImgPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16).to(device)
img2img.safety_checker = None
inpaint = StableDiffusionInpaintPipeline.from_pretrained("runwayml/stable-diffusion-inpainting", torch_dtype=torch.float16).to(device)
inpaint.safety_checker = None

## Exercise 1: Style transfer (watercolor building)

In [ ]:
building_url = "https://images.pexels.com/photos/466685/pexels-photo-466685.jpeg"
building_img = Image.open(requests.get(building_url, stream=True).raw).convert("RGB").resize((512,512))

prompt = "a watercolor painting of a building"
for strength in [0.3, 0.6, 0.9]:
    out = img2img(prompt=prompt, image=building_img, strength=strength).images[0]
    print(f"Strength {strength}")
    display(out)

## Exercise 2: Remove an object (person)
Use a photo with a person, mask them out.

In [ ]:
person_url = "https://images.pexels.com/photos/614810/pexels-photo-614810.jpeg"
person_img = Image.open(requests.get(person_url, stream=True).raw).convert("RGB").resize((512,512))
# Create a rough mask over the person (center area)
mask = Image.new("L", (512,512), 0)
draw = ImageDraw.Draw(mask)
draw.rectangle((150, 100, 362, 460), fill=255)

removed = inpaint(prompt="background, empty space, wall", image=person_img, mask_image=mask).images[0]
removed

## Exercise 3: Add a red balloon to a landscape

In [ ]:
landscape_url = "https://cdn.pixabay.com/photo/2015/04/23/22/00/tree-736885_1280.jpg"
landscape = Image.open(requests.get(landscape_url, stream=True).raw).convert("RGB").resize((512,512))
mask_balloon = Image.new("L", (512,512), 0)
draw = ImageDraw.Draw(mask_balloon)
draw.ellipse((300, 50, 380, 130), fill=255)  # top right area

with_balloon = inpaint(prompt="a bright red balloon", image=landscape, mask_image=mask_balloon).images[0]
with_balloon

## Exercise 4: Strength fixed, vary steps

In [ ]:
for steps in [10, 25, 50]:
    out = img2img(prompt="a fantasy landscape", image=landscape, strength=0.5, num_inference_steps=steps).images[0]
    print(f"Steps: {steps}")
    display(out)
print("Higher steps usually improve quality, but beyond 30-40 gains diminish.")

## Exercise 5: Automatic mask from segmentation (conceptual)
You could use a model like `facebook/detr-resnet-50` to detect "sky" region, generate mask, then inpaint with prompt "starry night".

```python
# Pseudocode:
# from transformers import DetrImageProcessor, DetrForObjectDetection
# processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
# model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
# inputs = processor(images=image, return_tensors="pt")
# outputs = model(**inputs)
# # Extract mask for class 'sky' (class id might be 33 in COCO)
# ...
# ```